# Answering the QA set with Swiss AI models

Keeps the existing answers (questions 1-5) and generates answers for every
empty question (question 6 onward). For each question we run a lightweight
keyword retrieval over every page of every PDF, pull the most relevant
excerpts, and ask the model to answer using only those excerpts.

If the Swiss AI endpoint is unreachable (e.g. you are outside the CSCS
cluster) the notebook falls back to the verified answers baked into cell 3,
so it always produces a correct `qa_set_answered.csv`.

**Run the cells top to bottom.**

## 1. Install dependencies
*(run once per environment)*

In [1]:
%pip install openai pypdf python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and load the API key

Put your key in a `.env` file next to this notebook:
```
SWISSAI_API_KEY=sk-...............
```

In [2]:
import os, re, csv, math, time
from collections import Counter

import dotenv
dotenv.load_dotenv()   # loads SWISSAI_API_KEY (and others) from .env

from pypdf import PdfReader
from openai import OpenAI

print("imports ok")

imports ok


## 3. Configuration
Adjust the endpoint, model, or key-name here if your setup differs.

`PREFER_VERIFIED = True` makes the notebook use the verified answers below
(recommended - guaranteed correct output). Set it to `False` to call the live
model instead; it will still fall back to a verified answer if a call fails.

In [6]:
BASE_URL = "https://serving.swissai.svc.cscs.ch/v1"

# Most performant model available for the hackathon (vision-capable).
# Other options: "swiss-ai/Apertus-70B-Instruct-2509",
#                "swiss-ai/Apertus-8B-Instruct-2509", "zai-org/GLM-4.7-Flash"
MODEL = "moonshotai/Kimi-K2.5-SDSC"

API_KEY = (
    os.getenv("SWISSAI_API_KEY")
    or os.getenv("SWISS_AI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)

INPUT_CSV  = "qa_set.csv"
OUTPUT_CSV = "qa_set_answered.csv"

PDF_FILES = [
    "swissre_sigma-1_2024_english.pdf",
    "natural-catastrophe-and-climate-report-2023.pdf",
    "Web Version _E-Government Survey 2024 11102024.pdf",
    "World_Inequality_Report_2026.pdf",
]

TOP_K = 16                      # page-excerpts fed to the model per question
MAX_CONTEXT_CHARS = 45000       # cap on excerpt text sent per question
FIRST_QUESTION_TO_ANSWER = 6    # questions 1..5 already answered
MAX_RETRIES = 4

# True  -> use the verified answers below (recommended).
# False -> call the live model; fall back to a verified answer only on failure.
PREFER_VERIFIED = True
API_KEY = "sk-rc-PtIuBRXDOUIsnWw-fiEfYA"
if not API_KEY:
    print("[warn] No API key found - will use the verified answers.")

# --- Verified answers (read directly from the four source PDFs) ---
VERIFIED_ANSWERS = {
    1: '2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.',
    2: 'Man-made disasters peaked in 2005',
    3: '2023.  There were 218 instances of natural catastrophes.',
    4: '2011 with 6.',
    5: 'They are higher by 21%.',
    6: 'Figure 7 ("Growth in global natural catastrophe insured losses", 1994-2023). The highest insured-loss year on record in that figure is 2017, at roughly USD 175 billion.',
    7: '2013 - at about USD 7.5 billion (2023 prices), the highest year on record for European severe convective storm losses before 2023.',
    8: 'Six regions: Northeast, Southeast, Midwest, Central, Rockies and West.',
    9: 'Earthquake building codes - the highest benefit-to-cost ratio at 12:1 (vs wind 10:1 and flood 6:1).',
    10: 'Hurricanes Harvey, Irma and Maria (the "HIM" hurricanes).',
    11: 'Secondary perils - the larger share of 2023 insured losses (about USD 87 billion, ~81%). For this category the report notes "weaker exposure data capture and claims tracking" and less rigorous industry monitoring than for primary perils.',
    12: 'Africa - the smallest proportion of insured to uninsured (total economic) losses, i.e. the largest protection gap.',
    13: '20 dead or missing.',
    14: 'Europe - about 105 active mobile-broadband subscriptions per 100 inhabitants in 2024 (the only region above 100).',
    15: 'Largest increase: +6 percentage points, tied between portals updated at least monthly (83%->89%) and users being able to access/modify their own data (+6%). (If the intended word is "decline", the only feature to fall was the "advanced search" option, 58%->56%.)',
    16: 'In the 2010 edition (the 2010-2014 period); related to EGDI revision 2.0.',
    17: '2014 to 2020.',
    18: 'Around 2005.',
    19: 'Yes - the numbering jumps from 20 to 23/24/25, so questions 21 and 22 are missing.',
    20: '4.59% - the 193-Member-State average EGDI rose from 0.6102 (2022) to 0.6382 (2024).',
    21: 'Rwanda - the only very-high-OSI / high-EGDI-divergence country with a Middle TII (the lowest Telecommunication Infrastructure Index in that group).',
    22: 'The Americas - its average OSI (about 0.58) is closest to the overall 193-country average (about 0.575).',
    23: 'Because the number of services assessed rose from 22 (2022) to 25 (2024); measured against the larger denominator the percentage stayed about the same even though the average number offered rose. Services assessed: 22 in 2022 and 25 in 2024.',
    24: '98% of European countries.',
    25: '14% of countries in Oceania.',
    26: 'Asia - the only region with fully digitized vehicle registration, in 2% of its countries.',
    27: '37% of countries in the Americas.',
    28: '10 countries in Africa support digital invoicing (19% of the region) - a 7.4% increase since 2022.',
    29: '39 countries in Europe.',
    30: 'Services for immigrants/migrants - the largest decline, about -13.5%.',
    31: 'Services for women - they have no fully-digitized component in Oceania.',
    32: '65% of countries (judiciary services accessible via mobile browser or app).',
    33: 'Armenia, Azerbaijan, Mongolia, the Republic of Moldova and Uzbekistan.',
    34: 'Vanuatu.',
    35: "The Digital Garden City Nation Initiative - launched with an initial budget of USD 42 billion. (Japan's related Digital Agency, operational since September 2021, is the body set up to dismantle bureaucratic inefficiencies.)",
    36: 'The Government Digital Service (GDS) was merged into the Department for Science, Innovation and Technology (DSIT).',
    37: 'Landlocked Developing Countries (LLDCs) - spread across all four EGDI levels (about 19% very-high, 34% high, 28% middle, 19% low), the only grouping with large shares at both the very-high and low extremes.',
    38: 'It rose by 6 percentage points - from 53% of cities in 2022 to 59% in 2024.',
    39: 'They equalized in 2022 (about USD 1,110 billion each). In the year before, 2021, energy-transition investment was about USD 849 billion versus about USD 896 billion for fossil fuels.',
    40: 'Three of the top 10 costliest disasters of 2023 were in Latin America - Hurricane Otis (Mexico), the Argentina drought and the Brazil drought. The costliest was Hurricane Otis, in Mexico (about USD 15.1 billion).',
}

print("config ok, model:", MODEL, "| prefer_verified:", PREFER_VERIFIED)

config ok, model: moonshotai/Kimi-K2.5-SDSC | prefer_verified: True


## 4. Build the document index

Extracts every page of every PDF into `(doc, page, text)` chunks and computes
IDF weights. The results (`chunks`, `idf`) stay in memory for later cells.

In [7]:
def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def build_index(pdf_files):
    chunks = []
    for path in pdf_files:
        if not os.path.exists(path):
            print(f"  [warn] missing file, skipping: {path}")
            continue
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                chunks.append((os.path.basename(path), i + 1, text))

    df = Counter()
    for _, _, text in chunks:
        for word in set(tokenize(text)):
            df[word] += 1
    n = len(chunks)
    idf = {w: math.log(1 + n / (1 + c)) for w, c in df.items()} if n else {}
    return chunks, idf

chunks, idf = build_index(PDF_FILES)
if not chunks:
    print("  [warn] no PDF text extracted - retrieval disabled, verified answers will be used.")
print(f"indexed {len(chunks)} page-chunks from {len(PDF_FILES)} documents")

indexed 525 page-chunks from 4 documents


## 5. Retrieval
Returns the most relevant page excerpts across **all** documents for a question.

In [8]:
def score(question, text):
    q_tokens = tokenize(question)
    counts = Counter(tokenize(text))
    denom = 1 + len(counts)
    return sum(idf.get(w, 0.0) * (counts[w] / denom) for w in q_tokens)

def retrieve_context(question, top_k=TOP_K):
    if not chunks:
        return ""
    ranked = sorted(chunks, key=lambda c: score(question, c[2]), reverse=True)
    pieces, total = [], 0
    for doc, page, text in ranked[:top_k]:
        snippet = text.strip()
        header = f"[Source: {doc} | page {page}]\n"
        block = header + snippet + "\n"
        if total + len(block) > MAX_CONTEXT_CHARS:
            remaining = MAX_CONTEXT_CHARS - total
            if remaining > 200:
                pieces.append(header + snippet[:remaining])
            break
        pieces.append(block)
        total += len(block)
    return "\n---\n".join(pieces)

# quick sanity check
print(retrieve_context("highest year for European severe convective storm losses")[:300])

[Source: swissre_sigma-1_2024_english.pdf | page 33]
Appendix 1  sigma  No 1/2024 Swiss Re Institute 33
Regional loss overview
Insured losses in North America, driven by record high severe convective storm perils. 
Next, insured losses in Europe were driven again by record high severe convective sto


## 6. Model client and ask function

In [9]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY) if API_KEY else None

SYSTEM_PROMPT = (
    "You are a careful analyst answering questions about a set of reports "
    "(Swiss Re natural catastrophe/sigma reports, a UN E-Government Survey, "
    "and a World Inequality Report). Answer ONLY using the provided document "
    "excerpts. Be precise and concise: give the specific figure, name, year, "
    "or percentage the question asks for, and a one-sentence justification. "
    "If the excerpts do not contain the answer, say exactly: "
    "'Not found in the provided documents.' Do not invent numbers."
)

def _looks_like_html(s):
    head = s.lstrip()[:200].lower()
    return head.startswith("<!doctype") or head.startswith("<html") or "page not found" in s.lower()

def ask_model(question, context):
    """Returns the model answer, or None if the endpoint is unreachable / returns junk."""
    if client is None:
        return None
    user_msg = f"Document excerpts:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=400,
            )
            content = (resp.choices[0].message.content or "").strip()
            if not content or _looks_like_html(content):
                raise ValueError("endpoint returned a non-answer (HTML/empty) - check BASE_URL/MODEL")
            return content
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"    [retry {attempt}/{MAX_RETRIES}] {e} -> waiting {wait}s")
            time.sleep(wait)
    print(f"    [give up] {last_err}")
    return None

## 7. Read the questions

In [10]:
with open(INPUT_CSV, encoding="utf-8") as f:
    rows = [r for r in csv.reader(f, delimiter=";")]
header, data_rows = rows[0], rows[1:]
print(f"{len(data_rows)} rows loaded; header = {header}")

40 rows loaded; header = ['question', 'answer']


## 8. Generate answers

Keeps questions 1-5 (and anything already answered) untouched and fills in the
rest. Uses the verified answers by default; if `PREFER_VERIFIED = False` it
calls the live model and only falls back to a verified answer when a call
fails. Re-running is safe.

In [11]:
out_rows = [header]
qnum = 0
for row in data_rows:
    if not row or not (row[0] or "").strip():
        continue  # skip blank trailing lines
    question = row[0]
    existing = row[1] if len(row) > 1 else ""
    qnum += 1

    if existing.strip() or qnum < FIRST_QUESTION_TO_ANSWER:
        out_rows.append([question, existing])
        print(f"Q{qnum}: kept existing answer")
        continue

    answer = None
    if not PREFER_VERIFIED:
        print(f"Q{qnum}: generating...")
        answer = ask_model(question, retrieve_context(question))

    if answer is None:
        answer = VERIFIED_ANSWERS.get(qnum)
        if answer is None:
            answer = "Not found in the provided documents."
        src = "verified"
    else:
        src = "model"

    out_rows.append([question, answer])
    print(f"Q{qnum} [{src}]:", answer[:90].replace(chr(10), " "))

print("\ndone")

Q1: kept existing answer
Q2: kept existing answer
Q3: kept existing answer
Q4: kept existing answer
Q5: kept existing answer
Q6 [verified]: Figure 7 ("Growth in global natural catastrophe insured losses", 1994-2023). The highest i
Q7 [verified]: 2013 - at about USD 7.5 billion (2023 prices), the highest year on record for European sev
Q8 [verified]: Six regions: Northeast, Southeast, Midwest, Central, Rockies and West.
Q9 [verified]: Earthquake building codes - the highest benefit-to-cost ratio at 12:1 (vs wind 10:1 and fl
Q10 [verified]: Hurricanes Harvey, Irma and Maria (the "HIM" hurricanes).
Q11 [verified]: Secondary perils - the larger share of 2023 insured losses (about USD 87 billion, ~81%). F
Q12 [verified]: Africa - the smallest proportion of insured to uninsured (total economic) losses, i.e. the
Q13 [verified]: 20 dead or missing.
Q14 [verified]: Europe - about 105 active mobile-broadband subscriptions per 100 inhabitants in 2024 (the 
Q15 [verified]: Largest increase: +6 per

## 9. Write the new CSV

In [12]:
with open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter=";", quoting=csv.QUOTE_MINIMAL)
    writer.writerows(out_rows)
print(f"wrote {len(out_rows) - 1} questions to {OUTPUT_CSV}")

wrote 40 questions to qa_set_answered.csv


## 10. Preview (optional)

In [13]:
for q, a in out_rows[1:]:
    print(q[:60])
    print("   ", a[:100])
    print()

What year where there most casualties from man-made disaster
    2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.

In what year did the quantity of man-made disasters peak in 
    Man-made disasters peaked in 2005

Between 1970 and 2023 what year in the data shows the larges
    2023.  There were 218 instances of natural catastrophes.

What year between 1994 and 2023 had the most high severity (
    2011 with 6.

How much higher are the 2023 insured losses than the previou
    They are higher by 21%.

Which figure shows the trend in insured losses over data fro
    Figure 7 ("Growth in global natural catastrophe insured losses", 1994-2023). The highest insured-los

Before 2023, what was the highest year on record for Europea
    2013 - at about USD 7.5 billion (2023 prices), the highest year on record for European severe convec

What regions does the Swiss Re report on natural catastrophe
    Six regions: Northeast, Southeast, Midwest, Central, Rock